In [2]:
%load_ext autoreload
%autoreload 2

In [23]:
%pip install numpy
%pip install pandas
%pip install pytest
%pip install requests
%pip install ipywidgets
%pip install jupyter_ui_poll

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrad

## Http fundermentals

**Using urllib to get geocoding into**

In [5]:
import json
import urllib.parse
import urllib.request

base_url = "https://geocoding-api.open-meteo.com/v1/search"
time_out=10
location='dresden'

try:
    params = {"name": location, "count": 1, "language": 'en', "format": 'json'}
    url = f"{base_url}?{urllib.parse.urlencode(params)}"
    with urllib.request.urlopen(url, timeout=time_out) as resp:
        results = json.load(resp).get("results", [])
        results[0]['latitude'], results[0]['longitude']

        print("Source: Open-Meteo Geocoding (geocoding-api.open-meteo.com/v1/search)\n" , f"Location:{location}\n", f"Latitude:{results[0]['latitude']}", f"Longitude:{results[0]['longitude']}")
except Exception as e:
    print(e)    



Source: Open-Meteo Geocoding (geocoding-api.open-meteo.com/v1/search)
 Location:dresden
 Latitude:51.05089 Longitude:13.73832


**Using Requests to get geocoding info**

In [6]:
import json
import requests

base_url = "https://geocoding-api.open-meteo.com/v1/search"
time_out=10
location='dresden'

try:
    params = {"name": location, "count": 1, "language": 'en', "format": 'json'}
    resp = requests.get(base_url, params=params, timeout=time_out)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    results[0]['latitude'], results[0]['longitude']

    print("Source: Open-Meteo Geocoding (geocoding-api.open-meteo.com/v1/search)\n" , f"Location:{location}\n", f"Latitude:{results[0]['latitude']}", f"Longitude:{results[0]['longitude']}")
except Exception as e:
    print(e)   

Source: Open-Meteo Geocoding (geocoding-api.open-meteo.com/v1/search)
 Location:dresden
 Latitude:51.05089 Longitude:13.73832


## Comparing urllib and requests

| | urllib | requests |
|---|---|---|
| How much code | More. I had to build the URL myself, open it, read the response and turn it into JSON. | Less. I pass the parameters in a dictionary and call `.json()` to get the data. |
| Handling errors | It raises different errors for different problems (`HTTPError` for a bad response, `URLError` for a network problem), so I have to catch each one. | `raise_for_status()` checks for a bad response, and `requests.RequestException` catches every kind of error in one go. |
| Sending headers | I have to create a `Request` object first to add headers. | I just pass `headers={...}` to `requests.get()`. |
| Encoding | I encode the URL with `urlencode` and decode the response myself. | It encodes the URL and decodes the response for me. |
| Installing | Nothing to install. It comes with Python. | I need to install it with `pip install requests`. |

urllib works without installing anything, but I had to write more code and handle more of the details myself. requests did the same job in fewer lines and was easier to read, so I would use it unless I couldn't install extra packages.

## Status code, the Content-Type header, and the response time 

In [7]:
import requests

base_url = "https://geocoding-api.open-meteo.com/v1/search"
time_out=10
location='dresden'

params = {"name":location, "count": 1, "language": "en", "format": "json"}
resp = requests.get(base_url, params=params, timeout=time_out)

print("Status code:  ", resp.status_code)
print("Content-Type: ", resp.headers["Content-Type"])
print(f"Response time: {resp.elapsed.total_seconds() * 1000:.0f} ms")

Status code:   200
Content-Type:  application/json; charset=utf-8
Response time: 895 ms


## Forecast retreival and display

In [ ]:
import ipywidgets as widgets
import requests
from IPython.display import display
from functools import partial

def geocode(location):
    base_url = "https://geocoding-api.open-meteo.com/v1/search"
    time_out=10
    params = {"name": location, "count": 10, "language": 'en', "format": 'json'}
    resp = requests.get(base_url, params=params, timeout=time_out)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    if not results:
        raise ValueError(f"No results found for {location!r}")

    selected = {"cords": None}

   # if more than 1 record
    if len(results) > 1:
        options = [("Choose a location...", None)]
        for r in results:
            label = ", ".join(p for p in [r['name'], r.get('admin1'), r.get('country')] if p)
            options.append((label, r))

        dropdown = widgets.Dropdown(options=options, description="Match:", layout=widgets.Layout(width="400px"))
        output = widgets.Output()
        dropdown.observe(partial(on_change, output=output, selected=selected), names="value")
        display(dropdown, output)
    else:
        r = results[0]
        selected["cords"] = [r['latitude'], r['longitude']]
        output = widgets.Output()
        with output:
            output.clear_output()
            print(f"Source: Open-Meteo Geocoding (geocoding-api.open-meteo.com/v1/search)\n"
                  f"Location: {location}\n"
                  f"Latitude: {r['latitude']}\n"
                  f"Longitude: {r['longitude']}")
        display(output)

def on_change(change, output, selected):
    """Listen for drop down changes"""
    with output:
        output.clear_output()
        r = change["new"]
        if r is not None:
            print(f"Source: Open-Meteo Geocoding (geocoding-api.open-meteo.com/v1/search)\n"
                              f"Location: {location}\n"
                              f"Latitude: {r['latitude']}\n"
                              f"Longitude: {r['longitude']}")

try:
    geocode('dresden')
except Exception as e:
    print(e)    

Dropdown(description='Match:', layout=Layout(width='400px'), options=(('Choose a location...', None), ('Dresde…

Output()